# TSMC SEC 6-K：接受日期后的下一 Qlib 交易日
三份已取得原始申报的来源时钟预检，不读市场收益或拟合。

In [1]:
import datetime,hashlib,json,os
from pathlib import Path
import pandas as pd,qlib
from qlib.data import D
root=Path.cwd().resolve();card=root/'research/stage1_tsmc_6k_next_session_clock_preflight_20261023';plan=json.loads((card/'plan.json').read_text());src=plan['source']
def sha(p):return hashlib.sha256(Path(p).read_bytes()).hexdigest()
repair=json.loads((card/'mechanical_provider_repair.json').read_text());assert repair['frozen_before_repair_execution'] and repair['original_plan_sha256']==sha(card/'plan.json') and sha(root/repair['replacement_provider']/'qlib_store_manifest.json')==repair['replacement_provider_manifest_sha256']
assert plan['frozen'] and sha(root/src['acceptance_result'])==src['acceptance_result_sha256'] and sha(root/src['provider']/'qlib_store_manifest.json')==src['provider_manifest_sha256'] and sha(root/'research/stage1_domain_heads_20261018/summary.json')==src['Stage1_summary_sha256']
source=json.loads((root/src['acceptance_result']).read_text());assert source['NAS_remote_and_scoped_restore_verified'] and not source['full_history_first_public_qualified']
checks=source['checks'];assert [x['accession'] for x in checks]==src['required_accessions'] and all(x['CIK']=='0001046179' for x in checks)
os.environ['MLFLOW_ALLOW_FILE_STORE']='true';qlib.init(provider_uri=str(root/repair['replacement_provider']),region='us',kernels=1)
calendar=pd.DatetimeIndex(D.calendar(start_time='2011-01-01',end_time='2026-10-09',freq='day')).normalize();assert calendar.is_unique and calendar.is_monotonic_increasing
assert calendar[0]<=pd.Timestamp('2011-07-28') and calendar[-1]>=pd.Timestamp('2026-07-17')
rows=[]
for item in checks:
 accepted=pd.Timestamp(datetime.datetime.strptime(item['accepted_timestamp_EDGAR_header'],'%Y%m%d%H%M%S'))
 later=calendar[calendar>accepted.normalize()];assert len(later)>0
 next_day=later[0];assert 0<(next_day.normalize()-accepted.normalize()).days<=7;decision_local=next_day.tz_localize('America/New_York')+pd.Timedelta(hours=9,minutes=46)
 ny=accepted.tz_localize('America/New_York').tz_convert('UTC');fixed=accepted.tz_localize(datetime.timezone(datetime.timedelta(hours=-5))).tz_convert('UTC')
 latest=max(ny,fixed);hours=(decision_local.tz_convert('UTC')-latest).total_seconds()/3600
 assert hours>=12 and next_day.normalize()>accepted.normalize()
 rows.append({'anchor':item['anchor'],'accession':item['accession'],'EDGAR_header_acceptance_naive':item['accepted_timestamp_EDGAR_header'],'NY_interpretation_UTC':ny.isoformat(),'fixed_EST_interpretation_UTC':fixed.isoformat(),'next_Qlib_trade_date':next_day.strftime('%Y-%m-%d'),'earliest_research_decision_ET':decision_local.isoformat(),'minimum_hours_from_later_acceptance_interpretation':hours,'source_text_identity_only':True})
result={'status':'three_SEC_6K_anchors_conservative_next_Qlib_session_clock_passed','plan_sha256':sha(card/'plan.json'),'mechanical_provider_repair_sha256':sha(card/'mechanical_provider_repair.json'),'invalid_attempt1_result':'attempt1_invalid_provider_result.json','anchors':rows,'minimum_hours_across_anchors':min(x['minimum_hours_from_later_acceptance_interpretation'] for x in rows),'SEC_source_NAS_archive_sha256':source['NAS_archive_sha256'],'SEC_source_NAS_scoped_restore_verified':True,'historical_first_public_clock_qualified':False,'full_2011_2026_guidance_inventory_qualified':False,'market_outcome_reads':0,'model_fits':0,'native_accounts':0}
(card/'result.json').write_text(json.dumps(result,indent=2)+'\n')
summary={'experiment':plan['experiment'],'decision':'conditional_three_anchor_next_session_clock_pass_after_provider_repair_full_history_pending','objective_achieved':False,'plan_sha256':result['plan_sha256'],'result':'result.json','source_anchors':len(rows),'minimum_hours_buffer':result['minimum_hours_across_anchors'],'model_fits':0,'native_accounts':0,'artifact_policy':{'policy':'slim_card_recorder_first','workflow_config':'workflow.yaml','recorder_only_artifacts':[]}}
(card/'summary.json').write_text(json.dumps(summary,indent=2)+'\n')
print(summary['decision'],[(x['accession'],x['next_Qlib_trade_date'],x['minimum_hours_from_later_acceptance_interpretation']) for x in rows])

[96525:MainThread](2026-10-23 05:41:38,145) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


2026/10/23 05:41:38 INFO mlflow.agent.hint: Load the `instrumenting-with-mlflow-tracing` skill at /Users/rziiiii/dev/workspace/qlib_venv/lib/python3.12/site-packages/mlflow/assistant/skills/instrumenting-with-mlflow-tracing/SKILL.md before writing any tracing code; it ships with this MLflow install. Set MLFLOW_DISABLE_AGENT_HINT=1 to silence this.


[96525:MainThread](2026-10-23 05:41:38,577) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.


[96525:MainThread](2026-10-23 05:41:38,578) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/dev/workspace/data/qlib/minute_information_pit_allocator_20261012/daily_prices')}


conditional_three_anchor_next_session_clock_pass_after_provider_repair_full_history_pending [('0001193125-11-199728', '2011-07-29', 26.17388888888889), ('0001564590-22-014381', '2022-04-18', 98.72416666666666), ('0001046179-26-000451', '2026-07-17', 25.004722222222224)]


/var/folders/4n/bc8v7c391nq228754jxg84_w0000gn/T/ipykernel_96525/2863791752.py:18: DeprecationWarning: The 'generic' unit for NumPy timedelta is deprecated, and will raise an error in the future. This includes implicit conversion of bare integers (e.g. `+ 1`). Please use a specific unit instead.
  next_day=later[0];assert 0<(next_day.normalize()-accepted.normalize()).days<=7;decision_local=next_day.tz_localize('America/New_York')+pd.Timedelta(hours=9,minutes=46)


## 中文结论
# TSMC SEC 6-K：接受后下一 Qlib 交易日来源时钟

复用三份已取得且 NAS 归档恢复通过的 TSMC 原始 6-K 接受头，仅计算下一实际 Qlib 交易日的 09:46 美东 Stage1 候选可用时点。分别得到：2011-07-28 接受→**2011-07-29**，2022-04-14 接受→**2022-04-18**（跨休市日），2026-07-16 接受→**2026-07-17**。即使将 SEC 头的时间按纽约当地和固定 EST 两种解释取较晚者，最小缓冲仍为 **25.00 小时**。这只是保守的来源时钟候选，不等于历史首收证明。

首次执行无效：原选的 Stage1 持仓 provider 日历从 2023 年才开始，错误把 2011、2022 两期映射到 2023-01-04。原计划、错误结果和代码 SHA 均保留；一次冻结机械修复改用原训练价格 Qlib 日历（2010–2026），并加七个自然日内必须出现下一交易日的门。累计仅两次 `D.calendar`、0 外部请求、0 市场收益读取、0 拟合、0 账户。

SEC [PDS 规范](https://www.sec.gov/files/info/edgar/specifications/pds_dissemination_spec.pdf)把接受时刻列为 EST，并说明公开分发与接受是不同步骤；这里采取下一实际交易日以免把申报当天直接当作已知。三份锚点通过不能证明 2011–2026 每季均有同义数值指引、当年完整原文或真实首次公开时间。决定：`conditional_three_anchor_next_session_clock_pass_after_provider_repair_full_history_pending`。完整历史来源、语义和 Stage1 信息门仍未完成，SOXX 目标未达成。
